[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/yellow/notebooks/yellow_data_curation.ipynb)

# Curating ACE inhibitors from four sources

**Yellow group · Hypertension**

Angiotensin-converting enzyme (ACE, sometimes called ACE-1) makes the hormone that
narrows blood vessels, and blocking it is one of the main ways hypertension is
treated (captopril, enalapril and lisinopril all work this way). This notebook
gathers every measurement of how strongly a compound blocks human ACE from ChEMBL,
BindingDB, PubChem and the group's own curation, keeps track of where each one came
from, and turns them into one table of molecules that a model can learn from.

## What you will do

- Load four downloads (ChEMBL, BindingDB, PubChem and the group's manual curation)
  and check they are all there.
- Standardise the molecules so that the same compound is recognised in every source.
- Put every value on one scale, calculate pActivity, and recover the compounds that
  only have a comment saying they are inactive.
- Find out which records are copies of each other, how much the sources overlap,
  and where they disagree.
- Build a final list of molecules, study how the choice of cutoff changes the labels,
  and repeat the curation for the rabbit enzyme.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "yellow"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Check the four downloads

This notebook works on files the group downloaded and put in its Drive folder. They
are copied into `data/` in this repository, so they are already there when you open
the notebook. Each source describes the same kind of experiment (how much compound is
needed to block ACE), but each writes it down differently, so it is worth knowing what
is inside each file, and how to get it again if you want fresher data.

We use two kinds of measurement, and keep them apart until the very end:

- **IC50** is the concentration of compound that halves the activity of the enzyme.
  It depends a little on how the experiment was run.
- **Ki** is the inhibition constant, a measure of how tightly the compound binds. It
  is more comparable between laboratories, but fewer papers report it.

For both, **a smaller number means a better compound**.

| File | Where to get it | What it looks like |
|---|---|---|
| `chembl_ace_human.csv` | Open the [ChEMBL page for CHEMBL1808](https://www.ebi.ac.uk/chembl/explore/target/CHEMBL1808) (human ACE), click on the number of activities, and export the table as CSV. Unzip it. | One row per measurement, of every type (IC50, Ki, % inhibition...). Separated by `;`. Has a `Comment` column and a `Data Validity Comment` column. |
| `bindingdb_ace_human.tsv` | On BindingDB's [targets by name](https://www.bindingdb.org/rwd/bind/ByTargetNames.jsp) page, find *Angiotensin-converting enzyme* for *Homo sapiens* (UniProt P12821) and click **TSV** in the Files column ([direct link](https://www.bindingdb.org/rwd/data/downloads/ptenK0/BDBpoly_2161.tsv)). | One row per compound and paper, separated by tabs. Each measurement type has its own column (`Ki (nM)`, `IC50 (nM)`), always in nM. `Curation/DataSource` says where BindingDB got the row. |
| `pubchem_ace_human.csv` | Open [this PubChem link](https://pubchem.ncbi.nlm.nih.gov/rest/pug/protein/accession/P12821/concise/CSV) in your browser; it downloads the bioactivity table for human ACE. | One row per compound and assay. Values in uM (`Activity Value [uM]`). Compounds are numbers (CIDs) with no structure: the notebook fetches the structures from PubChem. |
| `manual_ace_human.csv` | The group's table in the Drive folder `YellowTeam/Data` (*ace_data_new*): *File > Download > CSV*. | Your own format: `compound_id`, `smiles`, `standard_type`, `standard_value`, `standard_unit`, and `source` (where you took the row from). |

Section 7 also uses three files for the **rabbit** enzyme, obtained the same way:
`chembl_ace_rabbit.csv` (CHEMBL4074), `bindingdb_ace_rabbit.tsv`
([direct link](https://www.bindingdb.org/rwd/data/downloads/ptenK5000/BDBpoly_50000035.tsv))
and `pubchem_ace_rabbit.csv`
([link](https://pubchem.ncbi.nlm.nih.gov/rest/pug/protein/accession/P12822/concise/CSV)).

If you download new versions, name them exactly as above and put them in the
group's Drive folder; they are then copied into `data/` in the repository. The cell
below checks that all seven files are in `data/`.

In [ ]:
import os

import numpy as np
import pandas as pd
import stylia
from scripts import curation

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
RANDOM_SEED = 42
FILES = {"chembl": "data/chembl_ace_human.csv", "bindingdb": "data/bindingdb_ace_human.tsv",
         "pubchem": "data/pubchem_ace_human.csv", "manual": "data/manual_ace_human.csv"}
RABBIT_FILES = {"chembl": "data/chembl_ace_rabbit.csv",
                "bindingdb": "data/bindingdb_ace_rabbit.tsv",
                "pubchem": "data/pubchem_ace_rabbit.csv"}
wanted = list(FILES.values()) + list(RABBIT_FILES.values())

missing = [path for path in wanted if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f"not found in data/: {missing}. Run the setup cell again "
                            "to get the latest version of the repository.")
print("all seven files are in place")

Each source has its own reader in `scripts/curation.py`. They all return a table with the
same columns, so the four can be stacked into one. The PubChem reader also asks
PubChem for the structure of each compound and for who deposited each assay, which
takes about a minute the first time.

In [ ]:
records = pd.concat([
    curation.load_chembl(FILES["chembl"]),
    curation.load_bindingdb(FILES["bindingdb"]),
    curation.load_pubchem(FILES["pubchem"]),
    curation.load_manual(FILES["manual"]),
], ignore_index=True)
records.groupby("source").agg(records=("record_id", "size"),
                              structures=("smiles_in", "nunique"))

These are the columns every source now shares. `record_id` says exactly which row of
which file a record came from, and we carry it to the very end, so any number in the
final table can be traced back. `depositor` says who put the record into that
database, which matters in section 5.

In [ ]:
records.sample(5, random_state=RANDOM_SEED)

Not every record is an IC50 or a Ki. ChEMBL in particular keeps every kind of
readout (percentages of inhibition, ratios, log values...). This table shows the most
common types in each source.

In [ ]:
top_types = records["endpoint"].value_counts().head(8).index
pd.crosstab(records["endpoint"], records["source"]).loc[top_types]

We keep only IC50 and Ki. Percentages of inhibition cannot be turned into a
concentration without knowing the dose, and ratios compare two compounds rather than
measuring one.

> **Note:** ChEMBL stores one paper on ACE-inhibiting peptides as `logIC50` instead
> of IC50, so they are dropped here. BindingDB's full download has the same values
> converted to IC50, but the group's BindingDB file keeps only the 11 rows BindingDB
> did not take from ChEMBL, so those peptides are missing. Downloading the full
> BindingDB file would bring back about 90 molecules.

In [ ]:
ENDPOINTS = ["IC50", "Ki"]
records = records[records["endpoint"].isin(ENDPOINTS)].copy()
pd.crosstab(records["source"], records["endpoint"], margins=True)

## 2. Standardise the molecules

A molecule is written as a SMILES string, a line of text describing its atoms and
bonds. The same molecule can be written in many different SMILES, and databases
often store it as a salt (the molecule plus a counter-ion such as sodium). To
recognise the same compound across four sources we clean every structure the same
way and give it an **InChIKey**: a 27-character code that is identical for identical
molecules.

`standardize_smiles` removes salts and solvents, rewrites the molecule in a standard
form and computes its InChIKey. Structures it cannot read are left out.

In [ ]:
unique_smiles = records["smiles_in"].dropna().unique()
structures = curation.standardize_smiles(unique_smiles)
print(f"{len(unique_smiles):,} different SMILES -> {len(structures):,} standardised, "
      f"{structures['inchikey'].nunique():,} different molecules")

Now attach the InChIKey to every record and drop the records whose structure could
not be read. The table shows how many were lost in each source.

In [ ]:
before = records.groupby("source").size()
records = records.merge(structures, on="smiles_in", how="inner")
after = records.groupby("source").size()
pd.DataFrame({"before": before, "after": after, "lost": before - after})

Almost nothing is lost. What changes is the count of molecules: several SMILES often
turn out to be the same molecule once salts are removed. From here on a molecule is
its InChIKey.

In [ ]:
records.groupby("source").agg(structures_in=("smiles_in", "nunique"),
                              molecules=("inchikey", "nunique"))

## 3. Put every value on the same scale

The four sources use different units: ChEMBL and BindingDB mostly nM, PubChem uM, and
the manual table uM. We convert everything to **nanomolar (nM)** and then to a
**pActivity**, the same scale ChEMBL calls pChEMBL: minus the logarithm of the
concentration in molar. It turns small-is-good numbers into big-is-good ones and
spreads them evenly: 1 uM is 6, 10 nM is 8, 1 nM is 9.

First the **relation**. A record is either an exact measurement (`=`) or only a limit
(`>` 100 uM means "we tested up to 100 uM and saw nothing"). The sources write this in
several ways; `curation.RELATIONS` maps them onto `=`, `>` and `<`.

In [ ]:
records["value"] = pd.to_numeric(records["value"], errors="coerce")
relation = records["relation_raw"].fillna("").str.strip()
records["relation"] = relation.map(curation.RELATIONS)
pd.crosstab(records["source"], records["relation"].fillna("unknown"))

Next the units. Each source reports them in its own words; this is what we have.

In [ ]:
pd.crosstab(records["units"].fillna("none"), records["source"])

`to_nanomolar` multiplies each value by its unit's factor, and handles mass units
(ug/mL) with the molecular weight. Then we compute pActivity for every record that
has a positive value.

In [ ]:
records["value_nm"] = curation.to_nanomolar(records["value"], records["units"],
                                           records["mw"])
records["pactivity"] = curation.pactivity_or_nan(records["value_nm"])
records[["source", "endpoint", "relation", "value", "units", "value_nm",
         "pactivity"]].sample(5, random_state=RANDOM_SEED)

A quick check that the arithmetic is right: ChEMBL calculates its own pChEMBL for
exact measurements, and ours should agree.

In [ ]:
check = records[records["pchembl_db"].notna() & (records["relation"] == "=")]
agree = (check["pactivity"].round(2) - check["pchembl_db"].astype(float)).abs() <= 0.011
print(f"our pActivity matches ChEMBL's pChEMBL on {agree.mean():.1%} "
      f"of {len(check):,} records")

Now we **flag** the records we cannot use as a number, and say why. We do not delete
them yet: section 4 rescues some, and section 5 needs even the doubtful ones to
recognise copies of them.

- **no value**: the record has no number at all;
- **unit not convertible**: the unit is missing or is not a concentration;
- **implausible**: below 1 pM or above 1 M, almost certainly a typing error;
- **flagged by ChEMBL**: ChEMBL's curators marked the value as doubtful.

In [ ]:
reason = np.select(
    [records["value"].isna() | (records["value"] <= 0),
     records["value_nm"].isna() | records["relation"].isna(),
     ~records["value_nm"].between(1e-3, 1e9),
     records["validity"].notna()],
    ["no value", "unit not convertible", "implausible", "flagged by ChEMBL"],
    default="usable")
records["usable"] = reason == "usable"
pd.crosstab(reason, records["source"], margins=True)

## 4. Recover the records that only have a comment

Some records have no number, only a note such as *Not active*. They come from
screens where many compounds were tested at one concentration and most did nothing,
and they are worth keeping: they tell us which compounds are **inactive**, and good
inactives are rarer in the literature than actives.

Here are the comments on the records without a value.

In [ ]:
no_value = records[reason == "no value"]
no_value["comment"].fillna("(none)").value_counts()

`says_inactive` looks for words like *not active* or *inactive*. Comments such as
*Not Determined* are not evidence either way and stay unusable. The rescued records
get the relation `none`: they will be labelled inactive, but they have no pActivity.

> **Note:** For human ACE this rescues only a handful of records. For rabbit ACE, in
> section 7, it is more than a thousand, from a large screen called DrugMatrix.

In [ ]:
records["comment_inactive"] = curation.says_inactive(records["comment"])
rescued = (reason == "no value") & records["comment_inactive"]
records.loc[rescued, ["usable", "relation"]] = [True, "none"]
print(f"{rescued.sum()} records rescued from their comment")
records.groupby("source")["usable"].sum().rename("usable records")

## 5. Merge the sources: copies, overlap and disagreements

The four sources are **not independent**. BindingDB imports much of its data from
ChEMBL, PubChem receives its ACE data from ChEMBL and BindingDB, and the group's own
table was partly copied from both. If we simply pooled everything, the same
measurement would be counted two, three or four times, and the sources would look as
if they agreed far more than they really do.

### 5.1 Which records are copies

Each source says, in `depositor`, where a record came from. This table shows what
they say.

In [ ]:
records[records["usable"]].groupby(["source", "depositor"]).size().rename("records")

`mark_copies` checks each claim. ChEMBL is taken as the original, then BindingDB,
PubChem and the manual table, in that order. A record is the same measurement as
another when they share the molecule, the endpoint and the value (within 0.05 log
units, about 12%). Molecules are compared by the first 14 characters of the InChIKey,
which describe how the atoms are connected, because databases do not always agree on
the 3D arrangement (stereochemistry).

Every record gets a `copy_status`:

- **original**: it does not repeat anything we have;
- **confirmed copy**: it says it is a copy, and the original is there with the same value;
- **copy of a flagged record**: the same, but the original was flagged as doubtful;
- **copy, value differs**: it says it is a copy, the original has this molecule, but
  not this value, so one of the two was probably typed in wrong;
- **copy, original not found**: it says it is a copy, but we do not have the original;
- **duplicate**: it does not say where it came from, but an earlier source has the
  same molecule and value.

In [ ]:
records = curation.mark_copies(records, tolerance=0.05)
status = records[records["pactivity"].notna() | records["comment_inactive"]]
pd.crosstab(status["copy_status"], status["source"], margins=True)

Only the **originals** and the **copies whose original we do not have** bring new
information. Their values are the ones used from now on; everything else stays in
the table, and in `record_ids`, but does not count twice.

> **Note:** Most of PubChem's "original not found" records were deposited by ChEMBL,
> from old papers that ChEMBL files under an ACE target with no species. PubChem
> files the same assays under human ACE, so they reach us only through PubChem.

In [ ]:
records["trusted"] = records["usable"] & records["copy_status"].isin(curation.TRUSTED)
records.groupby("source")["trusted"].agg(["size", "sum"]).rename(
    columns={"size": "records", "sum": "trusted"})

### 5.2 How much the sources overlap

Now count molecules. `overlap_regions` works out how many molecules are in each
combination of sources, like the regions of a Venn diagram. We do it twice: counting
every usable record, and counting only the trusted ones.

In [ ]:
def molecule_sets(frame):
    return {s: set(frame.loc[frame["source"] == s, "inchikey"]) for s in curation.SOURCES}

every = curation.overlap_regions(molecule_sets(records[records["usable"]]))
trusted_only = curation.overlap_regions(molecule_sets(records[records["trusted"]]))
overlap = every.merge(trusted_only, on=["databases", "n_databases"], how="outer",
                      suffixes=("_all", "_trusted"))
overlap.fillna(0).astype({"molecules_all": int, "molecules_trusted": int})

With every record, most molecules appear in three or four sources, which looks
like strong agreement. With trusted records only, almost everything comes from
ChEMBL, and the other sources add a few hundred new molecules between them. The
plot shows the same thing.

In [ ]:
nc = stylia.NamedColors()
fig, axs = stylia.create_figure(1, 2)
for table, title, abc in [(every, "All usable records", "A"),
                          (trusted_only, "Trusted records only", "B")]:
    ax = axs.next()
    y = np.arange(len(table))
    ax.barh(y, table["molecules"], color=nc.yellow)
    ax.set_yticks(y)
    ax.set_yticklabels(table["databases"])
    ax.invert_yaxis()
    stylia.label(ax, xlabel="Molecules", ylabel="", title=title, abc=abc)

### 5.3 Conflicting values and possible mistakes

Two kinds of disagreement are worth flagging.

**Copies that differ from their original.** If a record says it was copied from
ChEMBL but has a different value, one of the two was transcribed wrongly. A gap of
exactly 3 log units (a factor of 1000) usually means someone confused nM and uM, so
we mark those as possible unit mistakes.

In [ ]:
original = records[(records["source"] == "chembl") & records["pactivity"].notna()]
differs = records[records["copy_status"] == "copy, value differs"]
pairs = differs.merge(original[["connectivity", "endpoint", "pactivity", "record_id"]],
                      on=["connectivity", "endpoint"], suffixes=("", "_chembl"))
pairs["gap"] = (pairs["pactivity"] - pairs["pactivity_chembl"]).abs()
pairs = pairs.sort_values("gap").drop_duplicates("record_id")
pairs["possible_unit_mistake"] = (np.isclose(pairs["gap"], 3, atol=0.15)
                                  | np.isclose(pairs["gap"], 6, atol=0.15))
pairs[["source", "record_id", "record_id_chembl", "endpoint", "pactivity",
       "pactivity_chembl", "gap", "possible_unit_mistake"]].sort_values("gap").tail(8)

The copy is kept out of the final values either way, because we trust the
original. But the molecule is flagged, so that anyone using the table knows its
value was recorded differently elsewhere.

**Independent measurements that disagree.** When the same molecule was measured
several times, in different papers, the values rarely match exactly. We flag
molecules whose trusted measurements span more than one log unit (a factor of ten).

In [ ]:
SPREAD_MAX = 1.0
exact = records[records["trusted"] & (records["relation"] == "=")]
spread = exact.groupby(["inchikey", "endpoint"])["pactivity"].agg(["size", "min", "max"])
spread = spread[spread["size"] >= 2]
spread["spread"] = spread["max"] - spread["min"]
print(f"{len(spread):,} molecule-endpoint pairs were measured more than once; "
      f"{(spread['spread'] > SPREAD_MAX).sum():,} of them disagree by more than "
      f"{SPREAD_MAX:g} log unit")
spread["spread"].describe().round(2)

We collect both flags per molecule, to attach them to the final table.

In [ ]:
flags = pd.DataFrame(index=records["inchikey"].unique())
flags["copy_differs"] = flags.index.isin(pairs["inchikey"])
flags["possible_unit_mistake"] = flags.index.isin(
    pairs.loc[pairs["possible_unit_mistake"], "inchikey"])
flags["conflicting_values"] = flags.index.isin(
    spread[spread["spread"] > SPREAD_MAX].index.get_level_values("inchikey"))
flags.sum().rename("molecules flagged")

## 6. Build the final list and choose a cutoff

### 6.1 One value per molecule

For each molecule and endpoint we take the **median** of the trusted exact
measurements; the median is used because one wrong number cannot drag it far. This is
the distribution we will cut.

In [ ]:
measured = curation.summarise_replicates(exact, ["inchikey", "endpoint"]).reset_index()
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for endpoint, color in zip(ENDPOINTS, [nc.yellow, nc.purple]):
    values = measured.loc[measured["endpoint"] == endpoint, "pactivity"]
    ax.hist(values, bins=50, range=(1, 12), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{endpoint} (n={len(values):,})")
ax.legend()
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title="Median pActivity per molecule, human ACE")

### 6.2 How the cutoff changes the labels

To train a classifier we must decide what counts as **active**. There is no correct
answer: a cutoff is a choice, and the fraction of actives moves a lot with it. The
table counts actives at several cutoffs, and how many molecules sit within half a log
unit of each one. Those near the line could fall on either side if they were measured
again, since we just saw that repeat measurements often differ by that much.

In [ ]:
rows = []
for cutoff_um in [0.1, 0.3, 1, 3, 10, 30]:
    line = curation.pactivity(cutoff_um * 1000)
    rows.append({"cutoff (uM)": cutoff_um, "pActivity": round(line, 2),
                 "active": int((measured["pactivity"] >= line).sum()),
                 "active %": round(100 * (measured["pactivity"] >= line).mean(), 1),
                 "within 0.5 log": int(((measured["pactivity"] - line).abs() < 0.5).sum())})
pd.DataFrame(rows)

A sanity check before choosing: the marketed ACE inhibitors should come out very
potent, whatever cutoff we pick. They are matched by the first block of their
InChIKey, so the check does not depend on how their stereochemistry was written.

In [ ]:
DRUGS = {"captopril": "C[C@H](CS)C(=O)N1CCC[C@H]1C(=O)O",
         "enalaprilat": "C[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O",
         "lisinopril": "NCCCC[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O"}
drugs = curation.standardize_smiles(DRUGS.values())
drugs["drug"] = list(DRUGS)
drugs["connectivity"] = drugs["inchikey"].str[:14]
measured["connectivity"] = measured["inchikey"].str[:14]
drugs.merge(measured, on="connectivity")[["drug", "endpoint", "inchikey_y", "n_exact",
                                         "pactivity"]]

Each drug is potent where it has many measurements. A drug can appear more than
once because the same atoms can be arranged in different 3D forms (stereoisomers),
which get different InChIKeys; the rarely measured forms can be much weaker, which
is exactly why stereochemistry matters.

We use **1 uM** (pActivity 6): it is a common choice for enzyme inhibitors and
gives a reasonably balanced set. Change `CUTOFF_UM` and re-run from here to see what
another choice does.

`collapse` now builds the table properly: medians of the trusted exact values, plus
the molecules that only have limits (`> 50 uM` proves inactivity at 1 uM, but `> 100 nM`
proves nothing and is dropped), plus the ones rescued from comments in section 4.

In [ ]:
CUTOFF_UM = 1.0
CUTOFF_NM = CUTOFF_UM * 1000
THRESHOLD = curation.pactivity(CUTOFF_NM)
per_endpoint = curation.collapse(records[records["usable"]], CUTOFF_NM)
per_endpoint["evidence"].value_counts()

### 6.3 Label and merge IC50 with Ki

Each molecule and endpoint gets its label. Limits and comments already say which
side they are on; measured values are compared with the threshold.

In [ ]:
per_endpoint["activity"] = np.select(
    [per_endpoint["evidence"].isin(["limit, inactive", "comment, inactive"]),
     per_endpoint["evidence"] == "limit, active"],
    [0, 1], default=(per_endpoint["pactivity"] >= THRESHOLD).astype(int))
pd.crosstab(per_endpoint["endpoint"], per_endpoint["activity"])

Now one row per molecule. A molecule with both an IC50 and a Ki gets the mean of the
two. If their labels disagree, the pooled value decides. Few molecules have both,
so this rarely matters.

In [ ]:
wide = per_endpoint.pivot(index="inchikey", columns="endpoint",
                          values=["pactivity", "activity"])
wide.columns = [f"{a}_{b.lower()}" for a, b in wide.columns]
labels = wide[["activity_ic50", "activity_ki"]]
wide["pactivity"] = wide[["pactivity_ic50", "pactivity_ki"]].mean(axis=1)
both_disagree = labels.notna().all(axis=1) & (labels.nunique(axis=1) > 1)
wide["activity"] = np.where(both_disagree, (wide["pactivity"] >= THRESHOLD).astype(int),
                            labels.bfill(axis=1).iloc[:, 0]).astype(int)
print(f"{labels.notna().all(axis=1).sum()} molecules have both; "
      f"{both_disagree.sum()} disagree on the label")
wide["activity"].value_counts().rename({0: "inactive", 1: "active"})

Finally, attach the provenance (which sources hold the molecule, which ones we
trusted, and every record id) and the flags from section 5.

In [ ]:
provenance = per_endpoint.groupby("inchikey").agg(
    smiles=("smiles", "first"),
    sources=("sources", lambda s: "+".join(sorted(set("+".join(s).split("+"))))),
    independent_sources=("independent_sources", lambda s: "+".join(
        sorted(set("+".join(s).split("+")) - {""}))),
    evidence=("evidence", lambda s: "+".join(sorted(set(s)))),
    record_ids=("record_ids", lambda s: ";".join(s)))
final = provenance.join(wide).join(flags)
final = final.sort_values("pactivity", ascending=False)
final.head()

Save the table. Colab deletes everything when it disconnects, so the cell below
saves two files and, in Colab, downloads both to your own computer (usually into your
`Downloads` folder):

- `ace_human_curated.csv`: the full table, one row per molecule.
- `ace_human_curated_smiles.csv`: only the SMILES, in one column headed `smiles`. This
  is the input format of the Ersilia Model Hub, so you can run its models on these
  molecules straight away.

Once they have downloaded, upload both files to the group's Drive folder
**Projects/YellowTeam/Data** so everyone works from the same files.

> **Note:** If nothing downloads, your browser may have blocked it. Look for a
> message near the address bar, allow downloads from Colab (including
> multiple files), and run the cell again.

In [ ]:
curation.save_output(final, "ace_human_curated.csv")
curation.save_output(curation.smiles_for_ersilia(final["smiles"]),
                     "ace_human_curated_smiles.csv", index=False)

> **Exercise:** How many of the final molecules rest on a single measurement, and
> how many carry one of the flags from section 5? Would you drop the flagged ones, or
> keep them and let the model see them? Try the cutoff at 10 uM and compare how many
> molecules change label.

## 7. Add the rabbit enzyme

For decades the standard ACE test used enzyme purified from **rabbit lungs**, so a
large part of the historical data is on rabbit ACE, not human. The two enzymes are
very similar, and a compound that blocks one usually blocks the other, but "usually"
is something we can check.

`curation.curate` runs sections 2 to 5 in one go on the three rabbit files. There is
no manual table for rabbit.

In [ ]:
rabbit_records, rabbit = curation.curate(
    [curation.load_chembl(RABBIT_FILES["chembl"]),
     curation.load_bindingdb(RABBIT_FILES["bindingdb"]),
     curation.load_pubchem(RABBIT_FILES["pubchem"])], CUTOFF_NM)
used = rabbit_records[rabbit_records["usable"]]
pd.crosstab(used["copy_status"], used["source"], margins=True)

Here section 4 matters much more: the DrugMatrix screen tested hundreds of
compounds against rabbit ACE at 10 uM and recorded only that they did nothing
("Inhibition < 50% @ 10 uM"). That makes them safe inactives for any cutoff up to
10 uM.

In [ ]:
rabbit["activity"] = np.select(
    [rabbit["evidence"].isin(["limit, inactive", "comment, inactive"]),
     rabbit["evidence"] == "limit, active"],
    [0, 1], default=(rabbit["pactivity"] >= THRESHOLD).astype(int))
pd.crosstab(rabbit["evidence"], rabbit["activity"], margins=True)

Now the real question: for molecules measured on both enzymes, do the values agree?
We compare the median pActivity of each molecule and endpoint on human and on rabbit
ACE.

In [ ]:
key = ["connectivity", "endpoint"]
human_values = measured.assign(connectivity=measured["inchikey"].str[:14])
rabbit_values = rabbit[rabbit["evidence"] == "measured"].assign(
    connectivity=rabbit["inchikey"].str[:14])
shared = human_values.merge(rabbit_values, on=key, suffixes=("_human", "_rabbit"))
gap = (shared["pactivity_human"] - shared["pactivity_rabbit"]).abs()
same_label = ((shared["pactivity_human"] >= THRESHOLD)
              == (shared["pactivity_rabbit"] >= THRESHOLD))
print(f"{len(shared):,} shared molecule-endpoints | median gap {gap.median():.2f} log | "
      f"same label {same_label.mean():.1%}")

The plot puts each shared molecule at its human value (across) and its rabbit value
(up). Points on the dashed line agree perfectly.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(shared["pactivity_human"], shared["pactivity_rabbit"], color=nc.yellow)
ax.plot([2, 11], [2, 11], color=nc.gray, linestyle="--")
ax.axvline(THRESHOLD, color=nc.pink, linestyle=":")
ax.axhline(THRESHOLD, color=nc.pink, linestyle=":")
stylia.label(ax, xlabel="Human ACE pActivity", ylabel="Rabbit ACE pActivity",
             title="Same molecules, two enzymes")

Many points sit exactly on the dashed line. Two separate experiments on two
different enzymes almost never give the same number to two decimals, so these are
one value reported twice: a paper quoting an earlier result, or a rabbit experiment
filed under human ACE. Let's see which human records repeat a rabbit value.

In [ ]:
rabbit_exact = rabbit_records[rabbit_records["usable"] & (rabbit_records["relation"] == "=")
                              & rabbit_records["copy_status"].isin(curation.TRUSTED)]
twins = exact.merge(rabbit_exact[["connectivity", "endpoint", "pactivity"]],
                    on=["connectivity", "endpoint"], suffixes=("", "_rabbit"))
twins = twins[(twins["pactivity"] - twins["pactivity_rabbit"]).abs() <= 0.05]
twins = twins.drop_duplicates("record_id")
print(f"{len(twins)} trusted human records repeat a rabbit value exactly")
twins.groupby(["source", "depositor"]).size().rename("records")

Leaving those twins aside, this is how far apart the two enzymes really are.

In [ ]:
independent = gap > 0.05
print(f"{independent.sum()} shared molecule-endpoints with different values | "
      f"median gap {gap[independent].median():.2f} log | "
      f"same label {same_label[independent].mean():.1%}")

How many new molecules would rabbit add to the human set?

In [ ]:
human_molecules = set(final.index.str[:14])
rabbit_molecules = set(rabbit["inchikey"].str[:14])
pd.Series({"human only": len(human_molecules - rabbit_molecules),
           "both": len(human_molecules & rabbit_molecules),
           "rabbit only": len(rabbit_molecules - human_molecules)}).rename("molecules")

> **Exercise:** Decide whether to pool the rabbit molecules with the human ones.
> Is the gap between the two enzymes smaller or larger than the spread between
> repeated measurements on the same enzyme (section 5.3)? If you pool them, keep a
> `species` column so a model, or a person, can tell them apart. Then look at the
> twins from the group's own table: open two or three of the papers and check which
> enzyme was really used.

## Summary

- You loaded four sources for human ACE and put them into one table in which every
  record keeps its source and its original id.
- After standardising the molecules and converting every value to pActivity, most of
  what BindingDB, PubChem and the manual table hold turned out to be copies of ChEMBL.
  The truly new molecules are about 270: mostly the group's own curation, and old
  ChEMBL papers that only PubChem files under human ACE.
- Copies that disagree with their original and molecules whose repeated measurements
  disagree are flagged, not hidden. Moving the cutoff from 0.1 to 10 uM moves the share
  of actives from about 43% to 77%.
- Rabbit ACE adds over a thousand molecules, most of them DrugMatrix inactives, and
  agrees with human ACE on the label about 83% of the time once repeated values are
  set aside.
- The curated table, `ace_human_curated.csv`, has one row per molecule with its
  pActivity, its active/inactive label at 1 uM, where it came from and its flags.
  `ace_human_curated_smiles.csv` holds its SMILES alone, ready for Ersilia.

**Next:** check that both files are in **Projects/YellowTeam/Data**, decide as a group
whether to add the rabbit molecules, and use the table to train a first model.